# 02 · Indicadores del mercado laboral, EPHPM 2025 (LXXXIII, julio 2025)

**Qué hace:** calcula los indicadores del mercado laboral con el factor de expansión, los desagrega por cortes y los contrasta con lo publicado por el INE.

**Entrada:** `data/processed/ephpm_2025_limpia.parquet` (salida del notebook de limpieza).
**Salidas:** `data/indicadores/cortes_2025.parquet|csv` (para Power BI) y `docs/contraste_ine_2025.csv`.

**Reglas del notebook**
1. Las definiciones son las que da el INE en sus documentos. No se intenta reconstruir las decisiones detrás de sus cifras.
2. Toda diferencia entre lo publicado y lo calculado se registra en `docs/discrepancias.csv`.
3. Lo que no es definición del INE (por ejemplo la informalidad) se marca como *aproximación propia*.

## 1. Configuración

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

RAIZ = Path.cwd()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
assert (RAIZ / "data").exists(), f"No encuentro la carpeta data en {RAIZ}"

ENTRADA = RAIZ / "data" / "processed" / "ephpm_2025_limpia.parquet"
SALIDA = RAIZ / "data" / "indicadores"
SALIDA.mkdir(parents=True, exist_ok=True)
(RAIZ / "docs").mkdir(exist_ok=True)

ESPERADO = {"filas": 28714, "pet": 20953}   # cifras validadas en la limpieza
UMBRAL_N = 30                               # mínimo de desocupados en la muestra para considerar confiable la TDA de un corte

## 2. Carga y validación de la entrada

In [2]:
base = pd.read_parquet(ENTRADA)

COLUMNAS = ["departamento", "edad", "sexo", "area", "dominio", "educacion", "cond_act", "tipo_inac",
            "subocup", "categoria", "aguinaldo", "catorceavo", "horas_principal", "ytrab",
            "desea_mas_horas", "salmin", "factor"]
faltan = [c for c in COLUMNAS if c not in base.columns]
assert not faltan, f"Faltan columnas: {faltan}"
assert len(base) == ESPERADO["filas"], f"Filas: {len(base)}"
print(f"{len(base):,} filas | {base.shape[1]} columnas")

28,714 filas | 25 columnas


## 3. Banderas y grupos
La población en edad de trabajar (PET) son las personas de 15 años o más (INE, desde 2020) y se identifica como `cond_act` no nulo.

In [3]:
ESTADOS = {"Ocupado", "Desocupado", "Fuera de la fuerza"}
extra = set(base["cond_act"].dropna().unique()) - ESTADOS
assert not extra, f"Etiquetas de cond_act distintas a las esperadas: {extra}. Ajuste ESTADOS y las banderas."

base["pet"]         = base["cond_act"].notna()
base["ocupado"]     = base["cond_act"].eq("Ocupado")
base["desocupado"]  = base["cond_act"].eq("Desocupado")
base["pea"]         = base["ocupado"] | base["desocupado"]
base["inactivo"]    = base["cond_act"].eq("Fuera de la fuerza")
base["desalentado"] = base["pet"] & base["tipo_inac"].eq("Desalentados")

assert base["pet"].sum() == ESPERADO["pet"], f"PET: {base['pet'].sum()}"
suma = base["ocupado"].astype(int) + base["desocupado"].astype(int) + base["inactivo"].astype(int)
assert suma.eq(base["pet"].astype(int)).all(), "Ocupados + desocupados + inactivos no suman la PET"

base["grupo_edad"] = pd.cut(base["edad"], bins=[14, 24, 34, 44, 54, 64, 120],
                            labels=["15-24", "25-34", "35-44", "45-54", "55-64", "65 y más"])

GRUPO_EDUC = {
    "Ninguno": "Sin educación formal", "Programa de alfabetización": "Sin educación formal",
    "Pre-básica (1-3)": "Básica", "Básica (1-9)": "Básica",
    "Media/Diversificado (1-4)": "Media",
    "Técnico superior (1-3)": "Superior", "Superior no universitaria (1-4)": "Superior",
    "Superior universitaria (1-8)": "Superior", "Especialidad (1-3)": "Superior",
    "Maestria (1-2)": "Superior", "Doctorado (1-4)": "Superior",
}
sin_grupo = set(base["educacion"].dropna().astype(str)) - set(GRUPO_EDUC)
assert not sin_grupo, f"Niveles educativos sin grupo: {sin_grupo}"
base["educ_grupo"] = pd.Categorical(base["educacion"].astype(object).map(GRUPO_EDUC),
                                    categories=["Sin educación formal", "Básica", "Media", "Superior"], ordered=True)

## 4. Indicadores nacionales y por cortes
- **TDA** = desocupados / PEA · **TGP** = PEA / PET · **tasa de ocupación** = ocupados / PET.
- «Sin dato» se muestra como categoría propia y nunca se elimina.
- `tda_confiable` marca los cortes con menos de `UMBRAL_N` desocupados en la muestra: la TDA de esos cortes es inestable.

In [4]:
def indicadores(df):
    w = df["factor"]
    pet, pea = w[df["pet"]].sum(), w[df["pea"]].sum()
    ocu, des = w[df["ocupado"]].sum(), w[df["desocupado"]].sum()
    return {"pet": pet, "pea": pea, "ocupados": ocu, "desocupados": des,
            "inactivos": w[df["inactivo"]].sum(), "desalentados": w[df["desalentado"]].sum(),
            "tda_%": des / pea * 100 if pea else np.nan,
            "tgp_%": pea / pet * 100 if pet else np.nan,
            "t_ocup_%": ocu / pet * 100 if pet else np.nan,
            "n_pet": int(df["pet"].sum()), "n_desocupados": int(df["desocupado"].sum())}

def tabla_cortes(df, cortes):
    filas = []
    for corte in cortes:
        serie = df[corte].astype(object).where(df[corte].notna(), "Sin dato")
        if str(df[corte].dtype) == "category":
            orden = list(df[corte].cat.categories)
        else:
            orden = sorted(df[corte].dropna().unique())
        for cat in orden + ["Sin dato"]:
            g = df[serie == cat]
            if g["pet"].sum() == 0:          # categoría vacía (por ejemplo, edad sin dato)
                continue
            filas.append({"corte": corte, "categoria": cat, **indicadores(g)})
    return pd.DataFrame(filas)

CORTES = ["sexo", "grupo_edad", "departamento", "area", "dominio", "educ_grupo"]
NACIONAL = indicadores(base)
cortes = tabla_cortes(base, CORTES)
cortes["tda_confiable"] = cortes["n_desocupados"] >= UMBRAL_N

# Comprobación: cada corte debe sumar al total nacional
for corte, g in cortes.groupby("corte"):
    for col in ["pet", "pea", "ocupados", "desocupados", "inactivos", "desalentados"]:
        assert abs(g[col].sum() - NACIONAL[col]) < 1, (corte, col, g[col].sum(), NACIONAL[col])
    assert g["n_pet"].sum() == NACIONAL["n_pet"], corte

cortes.to_parquet(SALIDA / "cortes_2025.parquet", index=False)
cortes.to_csv(SALIDA / "cortes_2025.csv", index=False, encoding="utf-8-sig")
display(pd.Series(NACIONAL).round(1).to_frame("nacional"))
display(cortes.round(1))

,nacional
pet,7363421.3
pea,4286430.0
ocupados,4075415.4
desocupados,211014.6
inactivos,3076991.3
desalentados,119999.3
tda_%,4.9
tgp_%,58.2
t_ocup_%,55.3
n_pet,20953.0


,corte,categoria,pet,pea,ocupados,desocupados,inactivos,desalentados,tda_%,tgp_%,t_ocup_%,n_pet,n_desocupados,tda_confiable
0,sexo,Hombre,3346465.9,2556378.1,2462889.4,93488.7,790087.8,37657.3,3.7,76.4,73.6,9626,258,True
1,sexo,Mujer,4016955.4,1730051.9,1612526.0,117525.9,2286903.5,82342.0,6.8,43.1,40.1,11327,335,True
2,grupo_edad,15-24,1710093.0,841194.7,770418.3,70776.4,868898.3,34062.2,8.4,49.2,45.1,4922,221,True
3,grupo_edad,25-34,1474474.8,1067369.1,995092.7,72276.4,407105.8,21929.6,6.8,72.4,67.5,4164,176,True
4,grupo_edad,35-44,1194414.0,847789.6,817617.0,30172.5,346624.4,23288.5,3.6,71.0,68.5,3507,86,True
5,grupo_edad,45-54,1111443.7,752484.5,729744.6,22739.8,358959.3,22370.6,3.0,67.7,65.7,3157,66,True
6,grupo_edad,55-64,814769.8,453971.4,442319.9,11651.5,360798.5,13562.5,2.6,55.7,54.3,2292,33,True
7,grupo_edad,65 y más,1058225.9,323620.8,320222.8,3398.0,734605.1,4786.0,1.0,30.6,30.3,2911,11,False
8,departamento,Atlántida,374370.4,186978.4,175619.5,11358.8,187392.0,4184.3,6.1,49.9,46.9,705,14,False
9,departamento,Choluteca,391138.1,240917.5,226776.3,14141.1,150220.7,10534.7,5.9,61.6,58.0,991,38,True


## 5. Contraste con lo publicado por el INE
Las estimaciones propias usan **la definición literal del documento metodológico** del INE:

- **Subocupación por tiempo (sec. 10.1):** menos de 40 horas y desea trabajar más. Se mide con `horas_principal` (no hay variable de horas de la ocupación secundaria).
- **Subocupación por ingresos (sec. 10.2):** 40 horas o más e ingreso mensual por trabajo (`ytrab`) inferior al salario mínimo de referencia (`salmin`).
- **Ingreso medio:** promedio de `ytrab` entre los ocupados que declaran ingreso.
- **Informalidad:** *aproximación propia*, no es la definición del INE. A = no es empleado privado ni público. B = A más empleados privados sin aguinaldo ni catorceavo.

Las variables `subocup` (oficial en la base) se muestran junto a la regla del documento para ver la diferencia.

In [5]:
w, oc_, cat = base["factor"], base["ocupado"], base["categoria"]
n_oc = w[oc_].sum()
pct = lambda m: w[oc_ & m].sum() / n_oc * 100

ASALARIADOS = ["Empleado doméstico", "Empleado u obrero privado", "Empleado u obrero público"]

quiere  = base["desea_mas_horas"].eq("Sí")
vis_doc = (base["horas_principal"] < 40) & quiere
inv_doc = (base["horas_principal"] >= 40) & (base["ytrab"] < base["salmin"])

priv  = cat.eq("Empleado u obrero privado")
inf_A = ~cat.isin(["Empleado u obrero privado", "Empleado u obrero público"])
inf_B = inf_A | (priv & base["aguinaldo"].eq("No") & base["catorceavo"].eq("No"))

def ingreso_medio(mask=None):
    m = oc_ & base["ytrab"].notna()
    if mask is not None:
        m = m & mask
    return (base.loc[m, "ytrab"] * w[m]).sum() / w[m].sum()

filas = [
 # indicador, ronda INE, valor INE, estimación propia, definición propia, discrepancias
 ("PET",                       "jul-2025", 7363421, NACIONAL["pet"],         "cond_act no nulo (15 años o más)", ""),
 ("Fuerza de trabajo",         "jul-2025", 4286430, NACIONAL["pea"],         "ocupados + desocupados", ""),
 ("Ocupados",                  "jul-2025", 4075415, NACIONAL["ocupados"],    "cond_act = Ocupado", ""),
 ("Desocupados",               "jul-2025", 211015,  NACIONAL["desocupados"], "cond_act = Desocupado", ""),
 ("TDA %",                     "jul-2025", 4.9,     NACIONAL["tda_%"],       "desocupados / PEA", ""),
 ("TGP %",                     "jul-2025", 58.2,    NACIONAL["tgp_%"],       "PEA / PET", ""),
 ("Asalariados",               "jul-2025", 2441067, w[oc_ & cat.isin(ASALARIADOS)].sum(), "doméstico + privado + público", ""),
 ("Subocup. tiempo % (variable subocup)",   "jul-2025", 8.0,  pct(base["subocup"].eq("Subocupados por tiempo de trabajo")), "variable oficial de la base", ""),
 ("Subocup. ingresos % (variable subocup)", "jul-2025", 33.4, pct(base["subocup"].eq("Subocupados por ingreso")),           "variable oficial de la base", ""),
 ("Subocup. tiempo % (regla del documento)",   "jul-2025", 8.0,  pct(vis_doc), "menos de 40 h y desea más horas", "D01 D02 D03 D07"),
 ("Subocup. ingresos % (regla del documento)", "jul-2025", 33.4, pct(inv_doc), "40 h o más e ytrab < salmin",    "D01 D02 D04 D05 D06"),
 ("Ingreso medio mensual, nacional (L.)",       "jul-2025", 9682,  ingreso_medio(),                                "ytrab, ocupados con ingreso", "D22"),
 ("Ingreso medio mensual, urbano (L.)",         "jul-2025", 11725, ingreso_medio(base["area"].eq("Urbana")),       "ytrab, ocupados con ingreso", "D22"),
 ("Ingreso medio mensual, rural (L.)",          "jul-2025", 6681,  ingreso_medio(base["area"].eq("Rural")),        "ytrab, ocupados con ingreso", "D22"),
 ("Ingreso medio mensual, sector público (L.)", "jul-2025", 18483, ingreso_medio(cat.eq("Empleado u obrero público")),  "ytrab, empleados públicos", "D22"),
 ("Ingreso medio mensual, sector privado (L.)", "jul-2025", 9776,  ingreso_medio(cat.eq("Empleado u obrero privado")),  "ytrab, empleados privados", "D22"),
 ("Informalidad % A (piso, aproximación propia)", "nov-2024", np.nan, pct(inf_A), "no es empleado privado ni público", "D12 D14 D16 D21"),
 ("Informalidad % B (aproximación propia)",       "nov-2024", 74.8,   pct(inf_B), "A + privados sin aguinaldo ni catorceavo", "D12 D13 D14 D21"),
]
contraste = pd.DataFrame(filas, columns=["indicador", "ronda_INE", "valor_INE", "estimacion_propia",
                                         "definicion_propia", "discrepancias"])
contraste["diferencia"] = contraste["estimacion_propia"] - contraste["valor_INE"]
tol = np.where(contraste["valor_INE"] >= 1000, 1.0, 0.051)   # el INE publica cifras enteras o con un decimal
contraste["coincide"] = np.where(contraste["valor_INE"].isna(), "n/a",
                          np.where(contraste["diferencia"].abs() <= tol, "sí", "no"))
contraste = contraste.round(1)
contraste.to_csv(RAIZ / "docs" / "contraste_ine_2025.csv", index=False, encoding="utf-8-sig")
display(contraste)

,indicador,ronda_INE,valor_INE,estimacion_propia,definicion_propia,discrepancias,diferencia,coincide
0,PET,jul-2025,7363421.0,7363421.3,cond_act no nulo (15 años o más),,0.3,sí
1,Fuerza de trabajo,jul-2025,4286430.0,4286430.0,ocupados + desocupados,,-0.0,sí
2,Ocupados,jul-2025,4075415.0,4075415.4,cond_act = Ocupado,,0.4,sí
3,Desocupados,jul-2025,211015.0,211014.6,cond_act = Desocupado,,-0.4,sí
4,TDA %,jul-2025,4.9,4.9,desocupados / PEA,,0.0,sí
5,TGP %,jul-2025,58.2,58.2,PEA / PET,,0.0,sí
6,Asalariados,jul-2025,2441067.0,2441067.0,doméstico + privado + público,,0.0,sí
7,Subocup. tiempo % (variable subocup),jul-2025,8.0,8.0,variable oficial de la base,,0.0,sí
8,Subocup. ingresos % (variable subocup),jul-2025,33.4,33.4,variable oficial de la base,,-0.0,sí
9,Subocup. tiempo % (regla del documento),jul-2025,8.0,6.8,menos de 40 h y desea más horas,D01 D02 D03 D07,-1.2,no


## 6. Registro de discrepancias
El registro vive en `docs/discrepancias.csv` y se edita a mano (Excel o VS Code). Aquí solo se resume y se comprueba que cada discrepancia citada en el contraste esté registrada.

In [6]:
ruta = RAIZ / "docs" / "discrepancias.csv"
if ruta.exists():
    disc = pd.read_csv(ruta)
    display(disc.groupby(["estado", "prioridad"]).size().rename("n").reset_index())
    citadas = set(" ".join(contraste["discrepancias"]).split())
    print("IDs citados en el contraste y no registrados:", sorted(citadas - set(disc["id"])) or "ninguno")
else:
    print("Aún no existe docs/discrepancias.csv")

,estado,prioridad,n
0,Abierta,Alta,3
1,Abierta,Baja,3
2,Abierta,Media,7
3,Cerrada,Baja,3
4,Cerrada,Media,1
5,Por verificar,Baja,1


IDs citados en el contraste y no registrados: ['D21', 'D22']
